# 特征筛选

1. 在第一个代码单元配置数据路径、标签、分箱参数、筛选阈值和 OOT 切分；
2. 第二个单元列出样本表里可选的 Y 标签；
3. 运行第三个单元开始筛选；筛选完成后默认对最终候选批量绘图。
4. 分析图、图片索引、分期明细和自动诊断写入 `feature_screening_reports/analysis_{标签名}_{运行时间}/`；`PLOT_RESULTS=False` 可关闭绘图。

OOT 只按 `sample_datetime` 时间切分，默认取最近 30%，切点落在时间点边界上；时间列只有一个取值时会报错。切点只用训练期拟合，OOT 只用于评估。入选需同时满足 IV 阈值、单调、OOT 稳定（方向与 PSI）、缺失箱 IV 占比不超限。结果写入数据根目录下的 `feature_screening_reports/`，文件名带标签名。

In [ ]:
from pathlib import Path

import importlib
import feature_screening as fs

importlib.reload(fs)  # 修改 feature_screening.py 后重跑本单元即可生效

# ---------- 数据与标签 ----------
ROOT_DIR = Path(fs.DEFAULT_ROOT_DIR)  # 固定数据根目录，与 notebook 位置无关
SAMPLE_FILE = "samples.csv"
VARIABLE_FILE = "variable_bank_statement_surplus.csv"
TARGET = "y_fpd15_ovd"  # 可选 y_fpd15_ovd / y_fpd7_ovd / y_mob1_5plus 等
OBS_COL = None           # None 时按 TARGET 自动匹配对应 *_obs
FEATURE_PREFIXES = None  # None 表示除主键、样本表同名列外全部数值列；非数值列自动跳过

# ---------- 分箱参数 ----------
N_BINS = 5          # 最多普通箱数（缺失箱不占额度）
N_PREBINS = 20      # 最多等频预分箱数，越大候选切点越细、越慢
MIN_BIN_PCT = 0.05  # 每个普通箱至少占非缺失样本的比例

# ---------- 筛选阈值 ----------
MIN_IV = 0.02               # 入选 IV 阈值（有 OOT 时取训练与 OOT 的较小值）
MAX_MISSING_IV_SHARE = 0.5  # 缺失箱 IV 占整体 IV 的上限；None 不限制

DEDUP_CORR = 0.8  # 入选变量 WOE 相关去重阈值；None 关闭去重

# ---------- 分析绘图 ----------
PLOT_RESULTS = True  # 是否对去重后的最终候选批量绘图
PLOT_PDF = False     # 是否同时输出同内容 PDF
PLOT_DPI = 200       # PNG 分辨率，建议 200~300
TIME_FREQ = "week"   # 时间稳定性粒度：day / week / month

# ---------- 分析诊断阈值（仅报告提示，不改变筛选结果） ----------
DIAG_MIN_BIN_N = 30          # 箱样本量提示阈值
DIAG_MIN_BIN_GOOD = 5        # 箱好样本量提示阈值
DIAG_MIN_BIN_BAD = 5         # 箱坏样本量提示阈值
DIAG_MIN_PERIOD_N = 100      # 时间段样本量提示阈值
DIAG_MIN_COVERAGE = 0.99     # 覆盖率提示阈值
DIAG_REVIEW_COVERAGE = 0.90  # 覆盖率复查阈值
DIAG_MAX_BIN_IV_SHARE = 0.5  # 单箱 IV 贡献提示阈值
DIAG_SMALL_BIN_IV_SHARE = 0.3  # 小样本箱贡献提示阈值
DIAG_DIRECTION_TOL = 0.05    # 方向反转坏账率容差
DIAG_JUMP_BAD_RATE = 0.10    # 异常跳点坏账率变化阈值
DIAG_PSI_REVIEW = 0.5        # 期间 PSI 复查阈值
DIAG_IV_FLOOR = 1e-6         # 判定 IV≈0 的下限
# ---------- OOT 时间切分 ----------
OOT_START = None  # 指定 OOT 起始日期（如 "2026-10-01"），优先于 OOT_RATIO
OOT_RATIO = 0.3   # 按 sample_datetime 取最近 30% 作 OOT；None 关闭 OOT 评估
MAX_PSI = 0.25    # OOT 与训练集分箱占比的最大 PSI；超限判为不稳定

In [ ]:
import pandas as pd

sample_columns = pd.read_csv(ROOT_DIR / SAMPLE_FILE, nrows=0).columns
print("样本表可选 Y 标签：")
print([c for c in sample_columns if c.startswith("y_")])
print("变量表列数：", len(pd.read_csv(ROOT_DIR / VARIABLE_FILE, nrows=0).columns))

In [ ]:
data, summary, bin_detail, candidates = fs.run_feature_screening(
    ROOT_DIR,
    min_iv=MIN_IV,
    n_bins=N_BINS,
    n_prebins=N_PREBINS,
    min_bin_pct=MIN_BIN_PCT,
    sample_file=SAMPLE_FILE,
    variable_file=VARIABLE_FILE,
    target=TARGET,
    obs_col=OBS_COL,
    feature_prefixes=FEATURE_PREFIXES,
    oot_start=OOT_START,
    oot_ratio=OOT_RATIO,
    max_psi=MAX_PSI,
    max_missing_iv_share=MAX_MISSING_IV_SHARE,
    dedup_corr=DEDUP_CORR,
    plot_results=PLOT_RESULTS,
    plot_pdf=PLOT_PDF,
    plot_dpi=PLOT_DPI,
    time_freq=TIME_FREQ,
    diag_min_bin_n=DIAG_MIN_BIN_N,
    diag_min_bin_good=DIAG_MIN_BIN_GOOD,
    diag_min_bin_bad=DIAG_MIN_BIN_BAD,
    diag_min_period_n=DIAG_MIN_PERIOD_N,
    diag_min_coverage=DIAG_MIN_COVERAGE,
    diag_review_coverage=DIAG_REVIEW_COVERAGE,
    diag_max_bin_iv_share=DIAG_MAX_BIN_IV_SHARE,
    diag_small_bin_iv_share=DIAG_SMALL_BIN_IV_SHARE,
    diag_direction_tol=DIAG_DIRECTION_TOL,
    diag_jump_bad_rate=DIAG_JUMP_BAD_RATE,
    diag_psi_review=DIAG_PSI_REVIEW,
    diag_iv_floor=DIAG_IV_FLOOR,
)

summary.head(30)